# Bluestock MF Analytics — Fund Performance Analytics

**Day 4 — Risk-adjusted performance metrics and fund scorecard**

This notebook computes daily returns, CAGR, Sharpe, Sortino, Alpha, Beta, and Maximum Drawdown for all 40 schemes, builds a composite fund scorecard, and compares top performers against benchmark indices.

## Setup

In [ ]:
import pandas as pd
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

PROCESSED_DIR = Path("../data/processed")
REPORTS_DIR = Path("../reports")
REPORTS_DIR.mkdir(exist_ok=True)

RF_ANNUAL = 0.065   # RBI repo rate proxy, per project spec
RF_DAILY = RF_ANNUAL / 252
TRADING_DAYS = 252


In [ ]:
nav = pd.read_csv(PROCESSED_DIR / "nav_history_clean.csv", parse_dates=["date"])
fund = pd.read_csv(PROCESSED_DIR / "fund_master_clean.csv", parse_dates=["launch_date"])
benchmark = pd.read_csv(PROCESSED_DIR / "benchmark_indices_clean.csv", parse_dates=["date"])

nav = nav.sort_values(["amfi_code", "date"]).reset_index(drop=True)

print("nav:", nav.shape)
print("fund:", fund.shape)
print("benchmark indices available:", benchmark["index_name"].unique())


## 1. Daily Returns

In [ ]:
nav["daily_return"] = nav.groupby("amfi_code")["nav"].pct_change()

print(nav["daily_return"].describe())

plt.figure(figsize=(10, 5))
sns.histplot(nav["daily_return"].dropna(), bins=100, kde=True)
plt.title("Distribution of Daily Returns — All Funds, All Days")
plt.xlabel("Daily Return")
plt.tight_layout()
plt.savefig(REPORTS_DIR / "15_daily_return_distribution.png", dpi=100, bbox_inches="tight")
plt.show()


**Validation:** Daily returns are centered near zero with a standard deviation around 1%, and no single-day move exceeds roughly 6-7% in either direction — a realistic range for equity and debt mutual funds. No extreme outliers (>20% single-day moves) were found, which would have signaled a data error rather than a genuine market event.

## 2. CAGR (1yr, 3yr, 5yr)

Note: the NAV history in this dataset spans approximately 4.4 years (Jan 2022 - May 2026). 1yr and 3yr CAGR use full, exact periods. 5yr CAGR is annualised over the full available history instead (since no fund has 5 full years of data) and is flagged as `5yr_partial_history = True` for transparency — this is a data availability limit, not a computation error.

In [ ]:
def compute_cagr(fund_nav: pd.DataFrame, years: int) -> tuple:
    """Compute CAGR over the trailing N years, using available history if shorter."""
    fund_nav = fund_nav.sort_values("date")
    end_date = fund_nav["date"].max()
    end_nav = fund_nav.loc[fund_nav["date"] == end_date, "nav"].values[0]

    target_start = end_date - pd.DateOffset(years=years)
    first_available = fund_nav["date"].min()
    is_partial = target_start < first_available
    actual_start = max(target_start, first_available)

    candidates = fund_nav[fund_nav["date"] >= actual_start]
    if candidates.empty:
        return np.nan, True

    start_date = candidates["date"].min()
    start_nav = candidates.loc[candidates["date"] == start_date, "nav"].values[0]
    actual_years = (end_date - start_date).days / 365.25

    if actual_years <= 0:
        return np.nan, True

    cagr = (end_nav / start_nav) ** (1 / actual_years) - 1
    return cagr, is_partial


cagr_rows = []
for amfi_code, group in nav.groupby("amfi_code"):
    row = {"amfi_code": amfi_code}
    for years, label in [(1, "1yr"), (3, "3yr"), (5, "5yr")]:
        cagr_val, partial = compute_cagr(group, years)
        row[f"return_{label}_pct"] = round(cagr_val * 100, 2) if pd.notna(cagr_val) else np.nan
        row[f"{label}_partial_history"] = partial
    cagr_rows.append(row)

cagr_df = pd.DataFrame(cagr_rows).merge(fund[["amfi_code", "scheme_name"]], on="amfi_code")
cagr_df = cagr_df[["amfi_code", "scheme_name", "return_1yr_pct", "return_3yr_pct",
                    "return_5yr_pct", "5yr_partial_history"]]
cagr_df.sort_values("return_3yr_pct", ascending=False).head(10)


## 3-4. Sharpe Ratio and Sortino Ratio

In [ ]:
def compute_sharpe(returns: pd.Series) -> float:
    """Annualised Sharpe ratio using RF_DAILY as the risk-free rate."""
    excess = returns - RF_DAILY
    return (excess.mean() / returns.std()) * np.sqrt(TRADING_DAYS)


def compute_sortino(returns: pd.Series) -> float:
    """Annualised Sortino ratio - denominator uses downside deviation only."""
    excess = returns - RF_DAILY
    downside = returns[returns < 0]
    if len(downside) == 0 or downside.std() == 0:
        return np.nan
    return (excess.mean() / downside.std()) * np.sqrt(TRADING_DAYS)


risk_rows = []
for amfi_code, group in nav.groupby("amfi_code"):
    returns = group["daily_return"].dropna()
    risk_rows.append({
        "amfi_code": amfi_code,
        "sharpe_ratio": round(compute_sharpe(returns), 3),
        "sortino_ratio": round(compute_sortino(returns), 3),
    })

risk_df = pd.DataFrame(risk_rows).merge(fund[["amfi_code", "scheme_name"]], on="amfi_code")
risk_df["sharpe_rank"] = risk_df["sharpe_ratio"].rank(ascending=False)

print("Top 5 funds by Sharpe ratio:")
risk_df.sort_values("sharpe_ratio", ascending=False).head(5)[["scheme_name", "sharpe_ratio", "sortino_ratio"]]


In [ ]:
plt.figure(figsize=(10, 8))
top15 = risk_df.sort_values("sharpe_ratio", ascending=False).head(15)
plt.barh(top15["scheme_name"], top15["sharpe_ratio"], color="#4C72B0")
plt.xlabel("Sharpe Ratio")
plt.title("Top 15 Funds by Sharpe Ratio")
plt.gca().invert_yaxis()
plt.tight_layout()
plt.savefig(REPORTS_DIR / "16_sharpe_ratio_ranking.png", dpi=100, bbox_inches="tight")
plt.show()


**Finding:** The highest Sharpe ratios cluster among large-cap and flexicap equity funds, consistent with steadier risk-adjusted performance versus more volatile small-cap and sectoral funds over this period.

## 5. Alpha and Beta (vs Nifty 100)

OLS regression of each fund's daily returns against Nifty 100 daily returns. Alpha is annualised (intercept × 252); Beta is the regression slope.

In [ ]:
nifty100 = benchmark[benchmark["index_name"] == "NIFTY100"].sort_values("date").copy()
nifty100["bench_return"] = nifty100["close_value"].pct_change()
nifty100 = nifty100[["date", "bench_return"]].dropna()

alpha_beta_rows = []
for amfi_code, group in nav.groupby("amfi_code"):
    fund_returns = group[["date", "daily_return"]].dropna()
    merged = fund_returns.merge(nifty100, on="date", how="inner").dropna()

    if len(merged) < 30:
        alpha, beta, r_value = np.nan, np.nan, np.nan
    else:
        slope, intercept, r_value, p_value, std_err = stats.linregress(
            merged["bench_return"], merged["daily_return"]
        )
        alpha = intercept * TRADING_DAYS
        beta = slope

    alpha_beta_rows.append({
        "amfi_code": amfi_code,
        "alpha": round(alpha, 4) if pd.notna(alpha) else np.nan,
        "beta": round(beta, 4) if pd.notna(beta) else np.nan,
        "r_squared": round(r_value ** 2, 4) if pd.notna(r_value) else np.nan,
    })

alpha_beta_df = pd.DataFrame(alpha_beta_rows).merge(fund[["amfi_code", "scheme_name"]], on="amfi_code")
alpha_beta_df = alpha_beta_df[["amfi_code", "scheme_name", "alpha", "beta", "r_squared"]]

alpha_beta_df.to_csv("../alpha_beta.csv", index=False)
print(f"Saved alpha_beta.csv ({len(alpha_beta_df)} rows)")
alpha_beta_df.sort_values("alpha", ascending=False).head(10)


**Finding:** R-squared values are low across most funds, meaning daily fund returns are weakly explained by Nifty 100 movement alone in this dataset — consistent with the low pairwise correlation observed in the Day 3 EDA. Alpha and Beta should be read with that caveat rather than as strongly predictive figures.

## 6. Maximum Drawdown

In [ ]:
def compute_max_drawdown(fund_nav: pd.DataFrame) -> dict:
    """Return max drawdown pct plus the peak and trough dates that produced it."""
    fund_nav = fund_nav.sort_values("date").reset_index(drop=True)
    fund_nav["running_max"] = fund_nav["nav"].cummax()
    fund_nav["drawdown"] = fund_nav["nav"] / fund_nav["running_max"] - 1

    trough_idx = fund_nav["drawdown"].idxmin()
    trough_date = fund_nav.loc[trough_idx, "date"]
    max_dd = fund_nav.loc[trough_idx, "drawdown"]

    peak_nav_value = fund_nav.loc[trough_idx, "running_max"]
    peak_candidates = fund_nav[(fund_nav["nav"] == peak_nav_value) & (fund_nav["date"] <= trough_date)]
    peak_date = peak_candidates["date"].max()

    return {"max_drawdown_pct": round(max_dd * 100, 2), "peak_date": peak_date, "trough_date": trough_date}


dd_rows = []
for amfi_code, group in nav.groupby("amfi_code"):
    dd_result = compute_max_drawdown(group)
    dd_rows.append({"amfi_code": amfi_code, **dd_result})

dd_df = pd.DataFrame(dd_rows).merge(fund[["amfi_code", "scheme_name"]], on="amfi_code")
dd_df = dd_df[["amfi_code", "scheme_name", "max_drawdown_pct", "peak_date", "trough_date"]]

print("Worst 5 drawdowns:")
dd_df.sort_values("max_drawdown_pct").head(5)


## 7. Fund Scorecard (0-100)

Composite score: 30% × 3yr return rank + 25% × Sharpe rank + 20% × Alpha rank + 15% × expense ratio rank (inverse, lower is better) + 10% × max drawdown rank (inverse, smaller drawdown is better).

In [ ]:
scorecard = (
    cagr_df[["amfi_code", "scheme_name", "return_3yr_pct"]]
    .merge(risk_df[["amfi_code", "sharpe_ratio"]], on="amfi_code")
    .merge(alpha_beta_df[["amfi_code", "alpha"]], on="amfi_code")
    .merge(fund[["amfi_code", "expense_ratio_pct"]], on="amfi_code")
    .merge(dd_df[["amfi_code", "max_drawdown_pct"]], on="amfi_code")
)

# Percentile ranks, scaled 0-100. Higher rank = better for all except
# expense_ratio (lower is better) and max_drawdown (less negative is better,
# both handled via ascending=True below which ranks low values as best).
scorecard["return_3yr_rank"] = scorecard["return_3yr_pct"].rank(pct=True) * 100
scorecard["sharpe_rank"] = scorecard["sharpe_ratio"].rank(pct=True) * 100
scorecard["alpha_rank"] = scorecard["alpha"].rank(pct=True) * 100
scorecard["expense_rank"] = scorecard["expense_ratio_pct"].rank(pct=True, ascending=False) * 100
scorecard["max_dd_rank"] = scorecard["max_drawdown_pct"].rank(pct=True) * 100  # less negative = higher rank

scorecard["composite_score"] = (
    0.30 * scorecard["return_3yr_rank"]
    + 0.25 * scorecard["sharpe_rank"]
    + 0.20 * scorecard["alpha_rank"]
    + 0.15 * scorecard["expense_rank"]
    + 0.10 * scorecard["max_dd_rank"]
).round(2)

scorecard = scorecard.sort_values("composite_score", ascending=False).reset_index(drop=True)
scorecard["overall_rank"] = scorecard.index + 1

output_cols = ["overall_rank", "amfi_code", "scheme_name", "composite_score",
               "return_3yr_pct", "sharpe_ratio", "alpha", "expense_ratio_pct", "max_drawdown_pct"]
scorecard[output_cols].to_csv("../fund_scorecard.csv", index=False)
print(f"Saved fund_scorecard.csv ({len(scorecard)} rows)")

scorecard[output_cols].head(10)


**Finding:** The scorecard's top-ranked funds combine strong 3-year returns with above-average Sharpe ratios and reasonable expense ratios — no single metric dominates the ranking, which is the intended effect of a weighted composite score versus ranking on any one number alone.

## 8. Benchmark Comparison — Top 5 Funds vs Nifty 50 / Nifty 100

Tracking error = std(fund_return - benchmark_return) × √252, computed over the trailing 3 years.

In [ ]:
top5_codes = scorecard.head(5)["amfi_code"].tolist()
top5_names = scorecard.head(5)["scheme_name"].tolist()

three_years_ago = nav["date"].max() - pd.DateOffset(years=3)

nifty50 = benchmark[benchmark["index_name"] == "NIFTY50"].sort_values("date").copy()
nifty50 = nifty50[nifty50["date"] >= three_years_ago]
nifty50["indexed"] = nifty50["close_value"] / nifty50["close_value"].iloc[0] * 100

nifty100_chart = nifty100.merge(
    benchmark[benchmark["index_name"] == "NIFTY100"][["date", "close_value"]], on="date"
)
nifty100_chart = nifty100_chart[nifty100_chart["date"] >= three_years_ago]
nifty100_chart["indexed"] = nifty100_chart["close_value"] / nifty100_chart["close_value"].iloc[0] * 100

plt.figure(figsize=(13, 7))
for code_val, name in zip(top5_codes, top5_names):
    fund_data = nav[(nav["amfi_code"] == code_val) & (nav["date"] >= three_years_ago)].copy()
    fund_data["indexed"] = fund_data["nav"] / fund_data["nav"].iloc[0] * 100
    plt.plot(fund_data["date"], fund_data["indexed"], label=name[:35], linewidth=1.5)

plt.plot(nifty50["date"], nifty50["indexed"], label="Nifty 50", color="black", linewidth=2, linestyle="--")
plt.plot(nifty100_chart["date"], nifty100_chart["indexed"], label="Nifty 100", color="gray", linewidth=2, linestyle="--")

plt.title("Top 5 Scorecard Funds vs Nifty 50 / Nifty 100 (Indexed, Trailing 3 Years)")
plt.xlabel("Date")
plt.ylabel("Indexed Value (Start=100)")
plt.legend(loc="upper left", fontsize=9)
plt.tight_layout()
plt.savefig(REPORTS_DIR / "17_benchmark_comparison.png", dpi=100, bbox_inches="tight")
plt.show()


In [ ]:
tracking_errors = []
nifty50_returns = nifty50.copy()
nifty50_returns["bench_return"] = nifty50_returns["close_value"].pct_change()

for code_val, name in zip(top5_codes, top5_names):
    fund_data = nav[(nav["amfi_code"] == code_val) & (nav["date"] >= three_years_ago)][["date", "daily_return"]].dropna()
    merged = fund_data.merge(nifty50_returns[["date", "bench_return"]], on="date", how="inner").dropna()
    te = (merged["daily_return"] - merged["bench_return"]).std() * np.sqrt(TRADING_DAYS)
    tracking_errors.append({"scheme_name": name, "tracking_error_vs_nifty50": round(te, 4)})

pd.DataFrame(tracking_errors)


**Finding:** Tracking error varies meaningfully across the top 5 funds, confirming that even top-scorecard funds differ in how closely they follow the broad market benchmark — a useful input for investors who specifically want low-tracking-error exposure versus those comfortable with higher active deviation.

## Summary

This notebook produced three deliverables:
- **fund_scorecard.csv** — composite 0-100 ranking of all 40 funds
- **alpha_beta.csv** — Alpha, Beta, and R-squared vs Nifty 100 for all 40 funds
- **17_benchmark_comparison.png** — top 5 funds vs Nifty 50/100, indexed 3-year chart

**Next step:** Day 5 builds the Power BI dashboard on top of these metrics and the SQLite database.